# Build your own controller

This notebook teaches **Option A**: a Python controller drives the production
`EngineSession` (Rust kernel) in a simple loop:

1. Read belief + pipeline from `session.snapshot()`
2. Choose an order with your `controller.order(ctx)`
3. Advance physics + filter with `session.step(order_qty)`

We compare a **naive base-stock** baseline and a small **tabular Q-learning**
starter against production **damped survival-weighted** ordering via
`session.act(policy="damped_sw")`.

**All simulator, logistics, filter, and economics knobs** live in the
**Simulation parameters** cell below (each with a comment). Controller-specific
hyperparameters are in a separate section. **tqdm** progress bars cover Q-learning
training and multi-seed evaluation.

Fixtures use `smoke_cool_shipments()` only (no Abdella parquet). This path does **not**
use `sim.episode.run_closed_loop_episode`.

## Setup

From the repo root with the Rust extension built:

```bash
uv sync --all-extras --python 3.11
uv run --python 3.11 maturin develop --release -m crates/voi_py/Cargo.toml
export BLUEBERRIES_VOI_BACKEND=rust
uv run jupyter lab notebooks/
```

Import the library helpers and confirm the Rust backend is available.

In [ ]:
%matplotlib inline

from __future__ import annotations

import os

os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from blueberries_voi.backend import rust_available, warn_fallback_once
from blueberries_voi.controller import (
    ControllerStepLog,
    EpisodeTotals,
    NaiveBaseStockController,
    TabularQLearningController,
    default_session_config,
    episode_totals_from_logs,
    run_controller_session,
)
from blueberries_voi.sim.profit import ProfitCosts
from blueberries_voi.sim.shipments import smoke_cool_shipments
from blueberries_voi.simulator import EngineSession

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})

warn_fallback_once()
if not rust_available():
    raise RuntimeError("Build blueberries_voi._core and set BLUEBERRIES_VOI_BACKEND=rust")
print("Rust backend ready.")

### Simulation parameters

Every simulator, logistics, filter, and economics knob is named below (with a comment
above each variable). `SESSION_CFG` and `PROFIT_COSTS` are built from those names only.

Smaller particle counts run faster in a notebook while keeping the same calendar and
filter structure. Production damped-SW uses `n_rollout_paths=0` (no nested rollout
scoring). **tqdm** shows progress during Q-learning training and multi-seed evaluation.

In [ ]:
# --- Reproducibility ---

# SEED: primary RNG seed for single-episode charts and Q-learning init
SEED = 42

# EVAL_SEEDS: paired multi-policy evaluation seeds (identical SESSION_CFG per arm)
EVAL_SEEDS = [11, 23, 37, 41, 53]

# --- Horizon ---

# N_DAYS: calendar days scored per controller episode
N_DAYS = 28

# TRAIN_EPISODES: tabular Q-learning training episodes
TRAIN_EPISODES = 8

# --- Logistics ---

# LEAD_TIME_DAYS: order-to-arrival lag in calendar days
LEAD_TIME_DAYS = 1

# DELIVERY_WEEKDAYS: weekday encoding Mon=0 .. Sun=6; MWF = (0, 2, 4)
DELIVERY_WEEKDAYS = (0, 2, 4)

# --- Shipments ---

# SHIPMENTS: inbound ASN trace (parquet-free smoke fixture)
SHIPMENTS = smoke_cool_shipments()

# --- Filter / belief wire ---

# ENABLE_FILTER: run particle filter on observations (False = open-loop truth)
ENABLE_FILTER = True

# BELIEF_SOURCE: belief fed to controller — "filter" (posterior) or "truth"
BELIEF_SOURCE = "filter"

# OBS_SCENARIO: observation-channel preset (P1 = production default)
OBS_SCENARIO = "P1"

# N_PARTICLES: particle filter size (lower = faster notebook)
N_PARTICLES = 64

# L_WIRE: max live lots on shelf (belief wire L)
L_WIRE = 10

# K_WIRE: freshness grid bins per lot (belief wire K)
K_WIRE = 4

# --- Act budgets (rollout scoring off in this notebook) ---

# H: protection horizon for act() candidate scoring
H = 7

# N_ROLLOUT_PATHS: nested rollout paths (0 = no rollout comparison)
N_ROLLOUT_PATHS = 0

# CANDIDATE_CASE_RADIUS: ±cases around base-stock candidate order
CANDIDATE_CASE_RADIUS = 1

# --- Damped SW act kwargs ---

# SW_ALPHA: survival-weight tail quantile for damped SW
SW_ALPHA = 0.9

# SW_RHO: demand damping factor for damped SW
SW_RHO = 0.8

# --- Economics (profit scoring) ---

# UNIT_MARGIN: dollar margin per unit sold
UNIT_MARGIN = 2.0

# WASTE_COST: dollar cost per spoiled unit
WASTE_COST = 1.5

# STOCKOUT_PENALTY: dollar penalty per lost sale (demand − sales)
STOCKOUT_PENALTY = 3.0

SESSION_CFG = default_session_config(
    shipments=SHIPMENTS,
    lead_time=LEAD_TIME_DAYS,
    delivery_weekdays=list(DELIVERY_WEEKDAYS),
    enable_filter=ENABLE_FILTER,
    belief_source=BELIEF_SOURCE,
    obs_scenario=OBS_SCENARIO,
    n_particles=N_PARTICLES,
    L=L_WIRE,
    K=K_WIRE,
    H=H,
    n_rollout_paths=N_ROLLOUT_PATHS,
    candidate_case_radius=CANDIDATE_CASE_RADIUS,
)

PROFIT_COSTS = ProfitCosts(
    unit_margin=UNIT_MARGIN,
    waste_cost=WASTE_COST,
    stockout_penalty=STOCKOUT_PENALTY,
)

### Controller parameters

The naive controller tops up to a fixed shelf target. The Q-learning starter explores
discrete order quantities on a coarse (weekday, on-hand bin) grid.

In [ ]:
# Naive base-stock target (units on shelf + pipeline)
BASE_STOCK_TARGET = 48

# Case size for naive rounding (matches ModelParams default)
CASE_SIZE = 8

# Discrete order actions for Q-learning (units, not cases)
QL_ACTIONS = [0, 8, 16, 24, 32]

# ε-greedy exploration probability during training
QL_EPSILON = 0.2

# Q-learning step size
QL_LEARNING_RATE = 0.15

# Discount factor (0 = maximize immediate day profit)
QL_DISCOUNT = 0.0

# On-hand cap for binning tabular states
QL_MAX_ON_HAND = 80

# Number of on-hand bins for tabular states
QL_ON_HAND_BINS = 8

## Train tabular Q-learning

Each training episode resets the session, runs `run_controller_session`, and calls
`observe` with **day profit** as the reward signal.

In [ ]:
ql = TabularQLearningController(
    QL_ACTIONS,
    epsilon=QL_EPSILON,
    learning_rate=QL_LEARNING_RATE,
    discount=QL_DISCOUNT,
    max_on_hand=QL_MAX_ON_HAND,
    on_hand_bins=QL_ON_HAND_BINS,
    seed=SEED,
)

train_profits: list[float] = []
for ep in tqdm(range(TRAIN_EPISODES), desc="Q-learning train"):
    session = EngineSession()
    session.init(SESSION_CFG, seed=SEED + ep)
    logs = run_controller_session(session, ql, N_DAYS, costs=PROFIT_COSTS)
    ep_profit = sum(log.day_profit for log in logs)
    train_profits.append(ep_profit)
    tqdm.write(f"train episode {ep + 1}/{TRAIN_EPISODES}: profit={ep_profit:.1f}")

ql.epsilon = 0.0  # greedy evaluation after training

## Multi-seed benchmark vs damped SW

After training, score **greedy Q-learning**, **naive base-stock**, and Rust
**damped SW** on the same evaluation seeds. Each seed fixes the demand path so
profit, spoilage, and stockout comparisons are **paired** (common random numbers).

The loop below retains **episode totals** (for histograms / violins / bars) and
**daily profit series** per policy per seed so later cumulative charts reuse this
run — no second full benchmark.

In [ ]:
naive = NaiveBaseStockController(target_units=BASE_STOCK_TARGET, case_size=CASE_SIZE)

POLICY_COLORS = {
    "Naive": "#4C72B0",
    "Q-learning": "#55A868",
    "Damped SW": "#C44E52",
}

ql_results: list[EpisodeTotals] = []
naive_results: list[EpisodeTotals] = []
sw_results: list[EpisodeTotals] = []
# Daily profit series keyed by (policy label, seed) — reused by cumulative chart
benchmark_daily: dict[tuple[str, int], list[float]] = {}


def _run_controller_daily(
    controller,
    seed: int,
    *,
    policy_label: str,
) -> tuple[list[ControllerStepLog], EpisodeTotals]:
    session = EngineSession()
    session.init(SESSION_CFG, seed=seed)
    logs = run_controller_session(session, controller, N_DAYS, costs=PROFIT_COSTS)
    totals = episode_totals_from_logs(
        logs, PROFIT_COSTS, seed=seed, policy_label=policy_label
    )
    return logs, totals


def _run_damped_sw_daily(seed: int) -> tuple[list[ControllerStepLog], EpisodeTotals]:
    session = EngineSession()
    session.init(SESSION_CFG, seed=seed)
    logs: list[ControllerStepLog] = []
    for _ in range(N_DAYS):
        delta = session.act(policy="damped_sw", alpha=SW_ALPHA, rho=SW_RHO)
        logs.append(ControllerStepLog.from_delta(delta, costs=PROFIT_COSTS))
    totals = episode_totals_from_logs(
        logs, PROFIT_COSTS, seed=seed, policy_label="Damped SW"
    )
    return logs, totals


for seed in tqdm(EVAL_SEEDS, desc="Benchmark seeds"):
    ql_logs, ql_totals = _run_controller_daily(ql, seed, policy_label="Q-learning")
    ql_results.append(ql_totals)
    benchmark_daily[("Q-learning", seed)] = [log.day_profit for log in ql_logs]

    naive_logs, naive_totals = _run_controller_daily(naive, seed, policy_label="Naive")
    naive_results.append(naive_totals)
    benchmark_daily[("Naive", seed)] = [log.day_profit for log in naive_logs]

    sw_logs, sw_totals = _run_damped_sw_daily(seed)
    sw_results.append(sw_totals)
    benchmark_daily[("Damped SW", seed)] = [log.day_profit for log in sw_logs]

    tqdm.write(
        f"seed={seed}: Q={ql_totals.profit:.1f} "
        f"naive={naive_totals.profit:.1f} SW={sw_totals.profit:.1f}"
    )

policy_groups = {
    "Q-learning": ql_results,
    "Naive": naive_results,
    "Damped SW": sw_results,
}

### Fig 1 — paired slopegraph (Q-learning vs damped SW)

Each gray segment connects the **same seed** under greedy Q-learning (left) and
damped SW (right). Upward slopes on profit mean Q-learning beat SW on that path.

In [ ]:
def _paired_metric_values(
    ql_rows: list[EpisodeTotals],
    sw_rows: list[EpisodeTotals],
    attr: str,
) -> tuple[list[float], list[float]]:
    return [getattr(row, attr) for row in ql_rows], [getattr(row, attr) for row in sw_rows]


BENCH_METRICS = [
    ("profit", "Total profit ($)"),
    ("waste", "Total spoilage (units)"),
    ("stockout", "Total stockout (units)"),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
x_positions = [0, 1]
for ax, (attr, ylab) in zip(axes, BENCH_METRICS, strict=True):
    ql_vals, sw_vals = _paired_metric_values(ql_results, sw_results, attr)
    for i in range(len(EVAL_SEEDS)):
        ax.plot(x_positions, [ql_vals[i], sw_vals[i]], color="0.8", lw=1, zorder=1)
        ax.scatter(
            x_positions[0], ql_vals[i], color=POLICY_COLORS["Q-learning"], s=30, zorder=2
        )
        ax.scatter(
            x_positions[1], sw_vals[i], color=POLICY_COLORS["Damped SW"], s=30, zorder=2
        )
    ax.set_xticks(x_positions)
    ax.set_xticklabels(["Q-learning", "Damped SW"])
    ax.set_ylabel(ylab)
    ax.set_title(f"Paired by seed (n={len(EVAL_SEEDS)})")
fig.suptitle("Q-learning vs damped SW — one line per seed")
fig.tight_layout()
plt.show()

### Fig 2 — paired deltas (Q-learning − damped SW)

Bar height is the per-seed difference. Positive profit bars favor Q-learning; lower
spoilage or stockout bars also favor Q-learning when the metric is a cost.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (attr, ylab) in zip(axes, BENCH_METRICS, strict=True):
    deltas = [
        getattr(q_row, attr) - getattr(sw_row, attr)
        for q_row, sw_row in zip(ql_results, sw_results, strict=True)
    ]
    colors = ["#55A868" if delta >= 0 else "#C44E52" for delta in deltas]
    ax.bar(range(len(EVAL_SEEDS)), deltas, color=colors)
    ax.axhline(0, color="0.3", lw=0.8)
    ax.set_xticks(range(len(EVAL_SEEDS)))
    ax.set_xticklabels([str(seed) for seed in EVAL_SEEDS], rotation=45, ha="right")
    ax.set_ylabel(f"Δ {ylab}")
    ax.set_xlabel("Eval seed")
fig.suptitle("Q-learning − damped SW")
fig.tight_layout()
plt.show()

### Fig 3 — distribution across seeds (violins)

Violin width shows how each policy spreads over `EVAL_SEEDS`. Compare medians and
spread when choosing a deployment default.

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (attr, ylab) in zip(axes, BENCH_METRICS, strict=True):
    data = [[getattr(row, attr) for row in policy_groups[name]] for name in policy_groups]
    parts = ax.violinplot(data, showmeans=True, showmedians=True)
    for body in parts["bodies"]:
        body.set_alpha(0.7)
    ax.set_xticks(np.arange(1, len(policy_groups) + 1))
    ax.set_xticklabels(list(policy_groups))
    ax.set_ylabel(ylab)
fig.suptitle("Policy spread across eval seeds")
fig.tight_layout()
plt.show()

### Fig 4 — distribution histograms (all three policies)

Overlaid histograms of episode totals from the benchmark loop above (no extra
simulation). Same palette as the paired and violin charts; `n=len(EVAL_SEEDS)` seeds
per policy.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (attr, ylab) in zip(axes, BENCH_METRICS, strict=True):
    for name, rows in policy_groups.items():
        values = [getattr(row, attr) for row in rows]
        ax.hist(
            values,
            bins=min(len(EVAL_SEEDS), 5),
            alpha=0.55,
            label=name,
            color=POLICY_COLORS[name],
            edgecolor="white",
            linewidth=0.6,
        )
    ax.set_ylabel("Count")
    ax.set_xlabel(ylab)
    ax.set_title(f"{ylab} (n={len(EVAL_SEEDS)} seeds)")
    ax.legend(fontsize=8)
fig.suptitle("Policy distributions across eval seeds")
fig.tight_layout()
plt.show()


## Cumulative profit (reuse benchmark daily logs)

Day-by-day cumulative profit for a **paired eval seed** — pulled from
`benchmark_daily` populated in the benchmark loop (no re-simulation). If `SEED` is
not in `EVAL_SEEDS`, one optional damped-SW day loop runs only for the `SEED`
overlay line.


In [ ]:
eval_seed = EVAL_SEEDS[0]

naive_daily = benchmark_daily[("Naive", eval_seed)]
ql_daily = benchmark_daily[("Q-learning", eval_seed)]
eval_sw_daily = benchmark_daily[("Damped SW", eval_seed)]

if SEED in EVAL_SEEDS:
    damped_daily = benchmark_daily[("Damped SW", SEED)]
else:
    sw_logs_seed, _ = _run_damped_sw_daily(SEED)
    damped_daily = [log.day_profit for log in sw_logs_seed]
    benchmark_daily[("Damped SW", SEED)] = damped_daily

print(f"Naive total profit (eval seed={eval_seed}): {sum(naive_daily):.1f}")
print(f"Q-learning total profit (eval seed={eval_seed}): {sum(ql_daily):.1f}")
print(f"Damped SW total profit (eval seed={eval_seed}): {sum(eval_sw_daily):.1f}")
if SEED != eval_seed:
    print(f"Damped SW total profit (seed={SEED}): {sum(damped_daily):.1f}")


### Cumulative profit comparison

In [ ]:
def cumulative(series: list[float]) -> list[float]:
    out: list[float] = []
    total = 0.0
    for x in series:
        total += x
        out.append(total)
    return out


fig, ax = plt.subplots()
days = list(range(1, len(naive_daily) + 1))
ax.plot(
    days,
    cumulative(naive_daily),
    label=f"Naive (eval seed={eval_seed})",
    color=POLICY_COLORS["Naive"],
)
ax.plot(
    days,
    cumulative(ql_daily),
    label=f"Q-learning (eval seed={eval_seed})",
    color=POLICY_COLORS["Q-learning"],
)
ax.plot(
    days,
    cumulative(eval_sw_daily),
    label=f"Damped SW (eval seed={eval_seed})",
    color=POLICY_COLORS["Damped SW"],
)
if SEED != eval_seed:
    ax.plot(
        days,
        cumulative(damped_daily),
        label=f"Damped SW (seed={SEED})",
        color=POLICY_COLORS["Damped SW"],
        ls="--",
        alpha=0.75,
    )
ax.set_xlabel("Day")
ax.set_ylabel("Cumulative day profit")
ax.set_title(f"Paired cumulative profit (eval seed={eval_seed})")
ax.legend()
fig.tight_layout()
plt.show()


### Mean profit across paired seeds

In [ ]:
by_policy: dict[str, list[float]] = {
    "Naive base-stock": [row.profit for row in naive_results],
    "Tabular Q-learning": [row.profit for row in ql_results],
    "Damped SW (act)": [row.profit for row in sw_results],
}

labels = list(by_policy.keys())
means = [mean(by_policy[label]) for label in labels]
colors = [POLICY_COLORS["Naive"], POLICY_COLORS["Q-learning"], POLICY_COLORS["Damped SW"]]

fig, ax = plt.subplots()
ax.bar(labels, means, color=colors)
ax.set_ylabel("Mean episode profit")
ax.set_title(f"Paired-seed benchmark ({len(EVAL_SEEDS)} seeds × {N_DAYS} days)")
fig.tight_layout()
plt.show()

## Next steps

- Swap `TabularQLearningController` for your own subclass of `ControllerTemplate`.
- Feed richer state from `ControllerContext.belief` (f-marginals per lot).
- Tune `BASE_STOCK_TARGET` or Q-learning actions against the same `run_controller_session` loop.
- Read ADR 0148 for why this path differs from `sim.episode.run_closed_loop_episode`.